# 03 — Cohort Analysis

**Purpose:** Track how customer retention changes over time, grouped by signup month.

---

## What is Cohort Analysis?

A **cohort** is a group of customers who all signed up in the same month.

Instead of looking at all customers as one big pool, cohort analysis asks:
> *"Of the 200 customers who signed up in January 2023, how many are still active 3 months later? 6 months? 12 months?"*

## Why Use It?

- Detects whether newer cohorts are being retained better than older ones
- Separates the effect of product improvements from mix-shifts
- Reveals at what point in the customer lifecycle most churn happens

## What is Retention Rate?

`retention_rate = (customers still active at month N) / (cohort_size) × 100`

Month 0 is always 100% — everyone just signed up. The number drops as customers churn.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import plotly.express as px

from data_processing import load_clean_data
from cohort_analysis import build_cohort_table, build_cohort_sizes, get_monthly_churn_trend
from visualization import plot_cohort_heatmap, plot_churn_trend

df = load_clean_data()
print(f'Loaded {len(df):,} customers')

## 1. Cohort Sizes — How Many Customers per Month?

In [ ]:
cohort_sizes = build_cohort_sizes(df)
print('Cohort sizes (first 12 months):')
print(cohort_sizes.head(12))

fig = px.bar(cohort_sizes.reset_index(), x='cohort', y='cohort_size',
             title='Number of New Customers by Signup Month',
             labels={'cohort': 'Signup Month', 'cohort_size': 'New Customers'})
fig.update_layout(xaxis_tickangle=45)
fig.show()

## 2. Build the Cohort Retention Table

In [ ]:
cohort_pivot = build_cohort_table(df)
print(f'Cohort table shape: {cohort_pivot.shape}')
print('First few cohorts, first 6 months:')
cohort_pivot.iloc[:6, :7].round(1)

## 3. Cohort Retention Heatmap

**How to read this:**
- Each **row** = customers who signed up in that month
- Each **column** = months since signup
- Each **cell** = % of that cohort still active
- **Green** = high retention, **Red** = high churn
- Month 0 is always 100%

In [ ]:
plot_cohort_heatmap(cohort_pivot).show()

## 4. Average Retention Curve

Averages across all cohorts to show the typical customer journey.

In [ ]:
avg_retention = cohort_pivot.mean(axis=0).reset_index()
avg_retention.columns = ['Month', 'Avg Retention %']
# Only first 13 months
avg_retention = avg_retention.head(13)

fig = px.line(avg_retention, x='Month', y='Avg Retention %', markers=True,
              title='Average Customer Retention Over Time (All Cohorts)',
              labels={'Avg Retention %': 'Retention (%)'})
fig.update_layout(yaxis_range=[0, 105], plot_bgcolor='white')
fig.add_hline(y=50, line_dash='dash', line_color='red', annotation_text='50% retention')
fig.show()

print(f'\nMonth 1 avg retention:  {avg_retention.iloc[1]["Avg Retention %"]:.1f}%')
print(f'Month 6 avg retention:  {avg_retention.iloc[min(6,len(avg_retention)-1)]["Avg Retention %"]:.1f}%')
print(f'Month 12 avg retention: {avg_retention.iloc[min(12,len(avg_retention)-1)]["Avg Retention %"]:.1f}%')

## 5. Monthly Churn Trend

In [ ]:
trend = get_monthly_churn_trend(df)
plot_churn_trend(trend).show()
print('\nFirst 6 months of trend data:')
trend.head(6)

## Business Interpretation

1. **Early churn is the biggest risk.** Most cohorts lose a significant percentage in months 1-3. This is where onboarding improvements have the most impact.

2. **Survivors are sticky.** Customers who make it past 6 months tend to stay much longer. Reaching that milestone should be a product goal.

3. **Cohort comparison.** If later cohorts show better retention than earlier ones, your product improvements are working. If they're similar, the problem is structural.

> **Action:** Focus retention investment on the first 90 days. A customer who doesn't find value in that window is unlikely to stay.